# Tremor Dominance Postural Instability Ratio

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import palettable.wesanderson as wes

import sys
sys.path.insert(0 , './../MDS-UPDRS_Analysis/')
import distributions
from distributions import *
import seaborn as sns

%load_ext autoreload
%autoreload 2

### Get Idiopathic ID's

In [2]:
meta = pd.read_csv('./../../data/PPMI/MO-LLM/PPMI_META_LINKAGE.csv')
idiopathic_ids = meta[np.logical_and(meta['Subgroup'] == 'Sporadic' , meta['CONCOHORT_DEFINITION'] == "Parkinson's Disease")]['PATNO'].unique()

### Specify Event IDs for which Genomic Data is Available

In [3]:
events = ['BL', 'V02' , 'V04' ,'V05',
          'V06' ,'V07' , 'V08' ,'V09' , 'V10' ,
          'V11' , 'V12' ,'V13' , 'V14' ,'V15' ,
          'V16' , 'V17' , 'V18','V19' ,'V20' ]

all_events = ['SC', 'BL', 'V01', 'V02', 'V03', 'V04', 'V05',
'V06', 'V07', 'V08', 'V09', 'V10', 'V11', 'V12',
'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
'ST', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25'
]

x_list = pd.DataFrame(np.arange(0 , 9.5 , 0.5) , index = events , columns=['response'])

## Section 1 - Pull in all relevant data

### MDS-UPDRS

#### Part 1

In [4]:
p1_1 = pd.read_csv('./../Assesment_Exams/MDS-UPDRS_Part_I_21Jan2025.csv')
p1_1_filt = p1_1.loc[: , ['PATNO' , 'EVENT_ID'] + [col for col in p1_1.columns if 'NP1' in col]]

p1_2 = pd.read_csv('./../Assesment_Exams/MDS-UPDRS_Part_I_Patient_Questionnaire_21Jan2025.csv')
p1_2_filt = p1_2.loc[: , ['PATNO' , 'EVENT_ID'] + [col for col in p1_2.columns if 'NP1' in col]]

p1_final = pd.merge(p1_1_filt , p1_2_filt , on = ['PATNO' , 'EVENT_ID'])
p1_final['NP1TOT'] = p1_final['NP1RTOT'] + p1_final['NP1PTOT']
p1_final = p1_final.drop(['NP1RTOT' , 'NP1PTOT'] , axis=1)
p1_final = p1_final.replace(101, np.nan).drop_duplicates()

p1_final = p1_final[p1_final['PATNO'].isin(idiopathic_ids)]

In [5]:
all_tmpt = pd.DataFrame({'EVENT_ID' : np.tile(events , len(p1_final['PATNO'].unique())) , 'PATNO' : np.repeat(p1_final['PATNO'].unique() , len(events))})

p1_final = p1_final[p1_final['EVENT_ID'].isin(events)]
p1_impute = pd.merge(all_tmpt , p1_final , on = ['PATNO' , 'EVENT_ID'] , how = 'outer')
p1_impute['EVENT_ID'] = pd.Categorical(p1_impute['EVENT_ID'] , categories=events , ordered=True)

for patno in p1_impute['PATNO'].unique() : 
    row_na = False
    df = p1_impute[p1_impute['PATNO'] == patno].copy()
    df['Temp_ID'] = df['EVENT_ID'].cat.codes.values
    for i in range(1, len(df) - 1):
        if df.iloc[i].isnull().any():  # Check if NaN exists in the row
            prev_row = df.iloc[i - 1]
            next_row = df.iloc[i + 1]
    
            # Check if the previous and next rows are consecutive
            if row_na : 
                if next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])
            else :
                if (prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID'] 
                        and next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID']):
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[[i - 1, i + 1]].mean(numeric_only=True))
                elif prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i - 1])
                elif next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])

            row_na = True
        else : 
            row_na = False
    
    df = df.drop('Temp_ID', axis=1)

    p1_impute[p1_impute['PATNO'] == patno] = df.copy()

p1 = p1_impute[np.logical_and(p1_impute['PATNO'].isin(idiopathic_ids) , p1_impute['EVENT_ID'].isin(events))]

#### Part 2

In [6]:
p2 = pd.read_csv('./../Assesment_Exams/MDS_UPDRS_Part_II__Patient_Questionnaire_21Jan2025.csv')
p2_final = p2.loc[: , ['PATNO' , 'EVENT_ID'] + [col for col in p2.columns if 'NP2' in col]]
p2_final = p2_final.replace(101, np.nan).drop_duplicates()

p2_final = p2_final[p2_final['PATNO'].isin(idiopathic_ids)]

In [7]:
all_tmpt = pd.DataFrame({'EVENT_ID' : np.tile(events , len(p2_final['PATNO'].unique())) , 'PATNO' : np.repeat(p2_final['PATNO'].unique() , len(events))})

p2_final = p2_final[p2_final['EVENT_ID'].isin(events)]
p2_impute = pd.merge(all_tmpt , p2_final , on = ['PATNO' , 'EVENT_ID'] , how = 'outer')
p2_impute['EVENT_ID'] = pd.Categorical(p2_impute['EVENT_ID'] , categories=events , ordered=True)

for patno in p2_impute['PATNO'].unique() : 
    row_na = False
    df = p2_impute[p2_impute['PATNO'] == patno].copy()
    df['Temp_ID'] = df['EVENT_ID'].cat.codes.values
    for i in range(1, len(df) - 1):
        if df.iloc[i].isnull().any():  # Check if NaN exists in the row
            prev_row = df.iloc[i - 1]
            next_row = df.iloc[i + 1]
    
            # Check if the previous and next rows are consecutive
            if row_na : 
                if next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])
            else :
                if (prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID'] 
                        and next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID']):
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[[i - 1, i + 1]].mean(numeric_only=True))
                elif prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i - 1])
                elif next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])

            row_na = True
        else : 
            row_na = False
    
    df = df.drop('Temp_ID', axis=1)

    p2_impute[p2_impute['PATNO'] == patno] = df.copy()

p2 = p2_impute[np.logical_and(p2_impute['PATNO'].isin(idiopathic_ids) , p2_impute['EVENT_ID'].isin(events))]

#### Part 3

In [8]:
p3 = pd.read_csv('./../Assesment_Exams/MDS-UPDRS_Part_III_21Jan2025.csv')
p3 = p3[p3['PDSTATE'].isin([np.nan , 'OFF'])]
p3_final = p3.loc[: , ['PATNO' , 'EVENT_ID' ,'NHY'] + [col for col in p3.columns if 'NP3' in col]]
p3_final = p3_final.replace(101, np.nan).drop_duplicates()

p3_final = p3_final[p3_final['PATNO'].isin(idiopathic_ids)]

p3_meta = p3.loc[ : , ['PATNO' , 'EVENT_ID' , 'PDTRTMNT' , 'PDSTATE' , 'HRPOSTMED' , 'HRDBSON', 'HRDBSOFF', 'PDMEDYN', 'DBSYN', 'ONOFFORDER', 'OFFEXAM', 'OFFNORSN', 'DBSOFFTM', 'ONEXAM', 'ONNORSN' ]]

In [9]:
all_tmpt = pd.DataFrame({'EVENT_ID' : np.tile(events , len(p3_final['PATNO'].unique())) , 'PATNO' : np.repeat(p3_final['PATNO'].unique() , len(events))})

p3_final = p3_final[p3_final['EVENT_ID'].isin(events)]
p3_impute = pd.merge(all_tmpt , p3_final , on = ['PATNO' , 'EVENT_ID'] , how = 'outer')
p3_impute['EVENT_ID'] = pd.Categorical(p3_impute['EVENT_ID'] , categories=events , ordered=True)

for patno in p3_impute['PATNO'].unique() : 
    row_na = False
    df = p3_impute[p3_impute['PATNO'] == patno].copy()
    df['Temp_ID'] = df['EVENT_ID'].cat.codes.values
    for i in range(1, len(df) - 1):
        if df.iloc[i].isnull().any():  # Check if NaN exists in the row
            prev_row = df.iloc[i - 1]
            next_row = df.iloc[i + 1]
    
            # Check if the previous and next rows are consecutive
            if row_na : 
                if next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])
            else :
                if (prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID'] 
                        and next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID']):
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[[i - 1, i + 1]].mean(numeric_only=True))
                elif prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i - 1])
                elif next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID'] : 
                    df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])

            row_na = True
        else : 
            row_na = False
    
    df = df.drop('Temp_ID', axis=1)

    p3_impute[p3_impute['PATNO'] == patno] = df.copy()

p3 = p3_impute[np.logical_and(p3_impute['PATNO'].isin(idiopathic_ids) , p3_impute['EVENT_ID'].isin(events))]

## Calculating Ratio - Stebbins et al.

In [10]:
mds_updrs_tmp = pd.merge(p1 , p2 , on = ['EVENT_ID' , 'PATNO'] )
mds_updrs = pd.merge(mds_updrs_tmp , p3 , on = ['EVENT_ID' , 'PATNO'])

In [28]:
mds_updrs_ratio.pivot_table(values = 'PIGD_Score', columns = 'EVENT_ID' , index = 'PATNO' , observed=False).to_csv('./../Labels/Stebbins_PIGD_Score_Scores.csv')

In [19]:
mds_updrs_ratio = mds_updrs[['PATNO' , 'EVENT_ID'] + tremor + pigd].reset_index(drop=True)

In [27]:
mds_updrs_ratio[pigd].mean(axis=1)

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
11585    NaN
11586    NaN
11587    NaN
11588    NaN
11589    NaN
Length: 11590, dtype: float64

In [26]:
tremor = ['NP2TRMR',
'NP3PTRMR',
'NP3PTRML',
'NP3KTRMR',
'NP3KTRML',
'NP3RTARU',
'NP3RTALU',
'NP3RTARL',
'NP3RTALL',
'NP3RTCON']

pigd = ['NP2WALK',
'NP2FREZ',       
'NP3GAIT',
'NP3FRZGT',
'NP3PSTBL']

mds_updrs_ratio = mds_updrs[['PATNO' , 'EVENT_ID'] + tremor + pigd].reset_index(drop=True)
#mds_updrs_ratio[tremor + pigd] = (mds_updrs_ratio[tremor + pigd] - mds_updrs_ratio[tremor + pigd].mean(axis=0))/mds_updrs_ratio[tremor + pigd].std(axis=0)

mds_updrs_ratio['Tremor_Score'] = mds_updrs_ratio[tremor].mean(axis=1 , skipna=False)
mds_updrs_ratio['PIGD_Score'] = mds_updrs_ratio[pigd].mean(axis=1 , skipna=False)

In [13]:
def td_pigd_ratio(td , pigd) : 
    if (td > 0) & (pigd == 0) : 
        if td < 1.15 : 
            return 1.15
        else : 
            return td
    elif (td ==0) & (pigd ==0) : 
        return 1
    elif np.isnan(td) | np.isnan(pigd) : 
        return np.nan
    else : 
        return td / pigd

In [14]:
mds_updrs_ratio['TD_PIGD_ratio'] = mds_updrs_ratio.apply(lambda row: td_pigd_ratio(row['Tremor_Score'] , row['PIGD_Score']) , axis=1)

In [17]:
mds_updrs_ratio.to_csv('./../Labels/TD_PIGD_Ratio.csv')

In [15]:
mds_updrs_ratio

,PATNO,EVENT_ID,NP2TRMR,NP3PTRMR,NP3PTRML,NP3KTRMR,NP3KTRML,NP3RTARU,NP3RTALU,NP3RTARL,NP3RTALL,NP3RTCON,NP2WALK,NP2FREZ,NP3GAIT,NP3FRZGT,NP3PSTBL,Tremor_Score,PIGD_Score,TD_PIGD_ratio
0,3001,BL,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0,1.15
1,3001,V02,1.0,2.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.8,0.0,1.15
2,3001,V04,2.0,2.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.8,0.0,1.15
3,3001,V05,1.0,1.0,1.0,2.0,1.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.15
4,3001,V06,1.0,1.0,1.0,3.0,2.0,1.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.3,0.0,1.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11585,163265,V16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11586,163265,V17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11587,163265,V18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11588,163265,V19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [138]:
tst = mds_updrs_ratio.groupby('PATNO').count()['PIGD_Score'] >= 5

In [139]:
tst

PATNO
3001       True
3002       True
3003       True
3006      False
3007      False
          ...  
160231     True
161236     True
162929    False
162994     True
163265    False
Name: PIGD_Score, Length: 610, dtype: bool

In [140]:
to_keep = tst[tst == 1].index

In [141]:
mds_updrs_ratio[mds_updrs_ratio['PATNO'].isin(to_keep)].to_csv('./../Labels/TD_PIGD_Ratio.csv')

In [133]:
mds_updrs_ratio[mds_updrs_ratio['PATNO'].isin(to_keep)]

,PATNO,EVENT_ID,NP2TRMR,NP3PTRMR,NP3PTRML,NP3KTRMR,NP3KTRML,NP3RTARU,NP3RTALU,NP3RTARL,NP3RTALL,NP3RTCON,NP2WALK,NP2FREZ,NP3GAIT,NP3FRZGT,NP3PSTBL,Tremor_Score,PIGD_Score,TD_PIGD_ratio
0,3001,BL,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.30,0.0,1.15
1,3001,V02,1.0,2.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.80,0.0,1.15
2,3001,V04,2.0,2.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.80,0.0,1.15
3,3001,V06,1.0,1.0,1.0,3.0,2.0,1.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.30,0.0,1.30
4,3001,V08,1.0,1.5,1.5,2.5,2.0,1.0,2.5,0.0,0.0,2.5,0.0,0.0,0.0,0.0,0.0,1.45,0.0,1.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3622,100017,V08,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.50,0.2,2.50
3623,100017,V10,2.0,0.0,1.0,1.0,2.0,2.0,2.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,0.0,1.40,0.2,7.00
3624,100017,V12,2.0,0.0,1.0,1.0,2.0,2.0,2.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,0.0,1.40,0.2,7.00
3625,100017,V14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Calculating Score - Skidmore et al.

In [83]:
mds_updrs_tmp = pd.merge(p1 , p2 , on = ['EVENT_ID' , 'PATNO'] )
mds_updrs = pd.merge(mds_updrs_tmp , p3 , on = ['EVENT_ID' , 'PATNO'])

In [79]:
p3.columns

Index(['EVENT_ID', 'PATNO', 'NHY', 'NP3SPCH', 'NP3FACXP', 'NP3RIGN',
       'NP3RIGRU', 'NP3RIGLU', 'NP3RIGRL', 'NP3RIGLL', 'NP3FTAPR', 'NP3FTAPL',
       'NP3HMOVR', 'NP3HMOVL', 'NP3PRSPR', 'NP3PRSPL', 'NP3TTAPR', 'NP3TTAPL',
       'NP3LGAGR', 'NP3LGAGL', 'NP3RISNG', 'NP3GAIT', 'NP3FRZGT', 'NP3PSTBL',
       'NP3POSTR', 'NP3BRADY', 'NP3PTRMR', 'NP3PTRML', 'NP3KTRMR', 'NP3KTRML',
       'NP3RTARU', 'NP3RTALU', 'NP3RTARL', 'NP3RTALL', 'NP3RTALJ', 'NP3RTCON',
       'NP3TOT'],
      dtype='object')

In [ ]:
features = {'NP1LTHD' : 4,
            'NP1FATG' : 1,
            'NP2SPCH' : 4,
            'NP2RISE' : 1,
            'NP2WALK' : 4,
            'NP3RISNG' : 6,
            'NP3POSTR' : 2
           }

mds_updrs_skid = mds_updrs[['PATNO' , 'EVENT_ID'] + list(features.keys())].reset_index(drop=True)
for item , score in features.items() : 
    print(item)
    feat_index = mds_updrs_skid[mds_updrs_skid[item] > 0].index   
    mds_updrs_skid.loc[feat_index , item] = score

mds_updrs_skid['Total'] = mds_updrs_skid.iloc[: , 2:].sum(axis=1)

In [76]:
mds_updrs_skid.to_csv('./../Labels/PIG_Skid_Score.csv')

In [90]:
mds_updrs_skid.iloc[[0,10,100,1000] , :]

,PATNO,EVENT_ID,NP1LTHD,NP1FATG,NP2SPCH,NP2RISE,NP2WALK,NP3RISNG,NP3POSTR,Total
0,3001,BL,0.0,1.0,0.0,0.0,0.0,0.0,2.0,3.0
10,3002,V02,0.0,1.0,4.0,1.0,4.0,0.0,2.0,12.0
100,3023,V02,4.0,0.0,0.0,1.0,4.0,0.0,2.0,11.0
1000,3314,V02,4.0,1.0,0.0,1.0,4.0,0.0,2.0,12.0
